## RobotWin open-loop eval（notebook）

按顺序执行下方 **code** 格（从上网下数第 1～4 格）：

| 格 | 内容 | 终端里大致对应 |
|---|------|----------------|
| **1** | `PROJECT_ROOT`、路径、`register_default_resolvers` | 基本无长日志 |
| **2** | `EVAL` + Hydra `compose` | 基本无长日志 |
| **3** | `load_openloop_model_for_eval` | `Loading model from ...`、`Loading Wan2.2...`、`MoT`、`PyTorch version` 等 **模型与权重** |
| **4** | `prepare_openloop_dataloader` + `run_openloop_evaluation` | `HTTP Request` / HuggingFace、`Resolving data files` 等 **数据集扫描**；随后是逐条推理与指标 |

**`EVAL["scope"]`**

- `"full_split"`：整个 `OPENLOOP.split`；`episode_indices=null`；`max_samples: null` 不截断。
- `"episodes"`：`episode_indices` 或单个 `episode_index`，与 `run_robotwin_openloop_episode.py` 一致。

内核 cwd 任意；第 1 格会 `chdir` 到仓库根，保证 `ckpt=` 等相对路径与 CLI 一致。**改数据范围：重跑 2→4；勿重跑 3 可省加载模型时间。**

**说明（Hydra / notebook）**：`compose()` 没有 Hydra 主进程，yaml 里带 `${hydra:...}` 的字段（默认的 `OPENLOOP.output_dir`）不能直接 `OmegaConf.resolve`。第 2 格在 compose 后会自动写成 `./evaluate_results/robotwin_openloop_episode/<task>/<时间戳>/`，或在 `EVAL` 里设 `"output_dir": "..."` 覆盖。

In [9]:
# === Cell 1 / 4：环境与路径（轻量，无模型、无数据集 IO）===
# 终端：通常只有 import；无 "Loading model" / "Resolving data files"。

import os
import sys
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import DictConfig, OmegaConf


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "configs" / "openloop_robotwin_episode.yaml").is_file():
            return p
        p = p.parent
    raise FileNotFoundError(
        "Could not find configs/openloop_robotwin_episode.yaml; "
        "cd to FastWAM repo root (or parent) and restart the kernel."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from fastwam.utils.config_resolvers import register_default_resolvers

register_default_resolvers()



In [10]:
# === Cell 2 / 4：Eval 范围 + Hydra compose（无模型加载、无数据集扫描）===
# 终端：基本无长日志；只打印本格末尾的 scope / ckpt 等。
#
# compose() 没有 Hydra 主进程，cfg 里若含 ${hydra:...}（如默认 OPENLOOP.output_dir）会报错。
# 下面在 compose 之后把 output_dir 写成具体路径；也可用 EVAL["output_dir"] 完全自定义。

from datetime import datetime


def _task_from_hydra_overrides(overrides: list[str]) -> str:
    for item in overrides:
        if item.startswith("task="):
            return item.split("=", 1)[1].strip()
    return "unknown_task"


# ---------------------------------------------------------------------------
# Eval scope (edit here)
# ---------------------------------------------------------------------------
# scope:
#   - "full_split": 使用整个 OPENLOOP.split，顺序遍历；OPENLOOP.episode_indices 强制为 null。
#     用 max_samples 控制最多跑多少条（None = 不限制，跑满整个 dataloader）。
#   - "episodes": 只评估列出的 episode（split 内 0-based），配合 frame_stride 在每条 episode 内采样。
#     填 episode_indices，例如 [0, 3]；若省略则使用 episode_index 单集。
EVAL = {
    "scope": "episodes",  # "full_split" | "episodes"
    "episode_index": 0,
    "episode_indices": [0],
    # full_split 下常用 None 跑全量；episodes 下可限制总 sample 数
    "max_samples": 8,
    # 可选：手写输出目录；不设则自动 ./evaluate_results/robotwin_openloop_episode/<task>/<时间戳>/
    # "output_dir": "./evaluate_results/robotwin_openloop_episode/my_run",
    "hydra_overrides": [
        "task=robotwin_uncond_3cam_384_1e-4",
        "ckpt=./checkpoints/fastwam_release/robotwin_uncond_3cam_384.pt",
        "OPENLOOP.dataset_stats_path=./checkpoints/fastwam_release/robotwin_uncond_3cam_384_dataset_stats.json",
        "OPENLOOP.split=train",
        "OPENLOOP.frame_stride=50",
        "OPENLOOP.predict_video=true",
        "OPENLOOP.save_video_samples=2",
    ],
}

GlobalHydra.instance().clear()
with initialize_config_dir(version_base="1.3", config_dir=str(PROJECT_ROOT / "configs")):
    cfg: DictConfig = compose(
        config_name="openloop_robotwin_episode",
        overrides=list(EVAL["hydra_overrides"]),
    )

scope = str(EVAL["scope"]).strip().lower()
if scope == "full_split":
    cfg.OPENLOOP.episode_indices = None
elif scope == "episodes":
    eps = EVAL.get("episode_indices")
    if eps is None:
        eps = [int(EVAL["episode_index"])]
    cfg.OPENLOOP.episode_indices = [int(x) for x in eps]
else:
    raise ValueError(f'EVAL["scope"] must be "full_split" or "episodes", got: {EVAL["scope"]!r}')

if "max_samples" in EVAL:
    cfg.OPENLOOP.max_samples = EVAL["max_samples"]

if EVAL.get("output_dir"):
    cfg.OPENLOOP.output_dir = str(EVAL["output_dir"])
else:
    _task = _task_from_hydra_overrides(EVAL["hydra_overrides"])
    _stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    cfg.OPENLOOP.output_dir = f"./evaluate_results/robotwin_openloop_episode/{_task}/{_stamp}"

print("scope:", scope)
print("output_dir:", cfg.OPENLOOP.output_dir)
print("episode_indices:", cfg.OPENLOOP.get("episode_indices"))
print("max_samples:", cfg.OPENLOOP.get("max_samples"))
print("ckpt:", cfg.ckpt)

scope: episodes
output_dir: ./evaluate_results/robotwin_openloop_episode/robotwin_uncond_3cam_384_1e-4/20260513_192918
episode_indices: [0]
max_samples: 8
ckpt: ./checkpoints/fastwam_release/robotwin_uncond_3cam_384.pt


In [11]:
# === Cell 3 / 4：只加载模型与 checkpoint（耗时长，调试后半段时不要重跑）===
# 终端对应片段（与 run_robotwin_openloop.py 一致）：
#   - "Loading model from ..."
#   - "Loading Wan2.2-TI2V-5B components..." / Skipping pretrained ... / Finished loading ...
#   - ActionDiT / MoT / "PyTorch version ... available."
# 下一格才会出现数据集相关的 HTTP / "Resolving data files"。

from fastwam.utils.logging_config import setup_logging

from experiments.robotwin.run_robotwin_openloop import load_openloop_model_for_eval

setup_logging()
model, ckpt_path, output_dir = load_openloop_model_for_eval(cfg)
print("output_dir:", output_dir)

05/13 [19:29:18] INFO     | >> Loading model from                                      ]8;id=831568;file:///data_all/xiangchengzhan/FastWAM/experiments/robotwin/run_robotwin_openloop.py\run_robotwin_openloop.py]8;;\:]8;id=646714;file:///data_all/xiangchengzhan/FastWAM/experiments/robotwin/run_robotwin_openloop.py#337\337]8;;\
                          /data_all/xiangchengzhan/FastWAM/checkpoints/fastwam_release                             
                          /robotwin_uncond_3cam_384.pt                                                             

                 INFO     | >> Loading Wan2.2-TI2V-5B components...                                   ]8;id=277099;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py\loader.py]8;;\:]8;id=204106;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py#152\152]8;;\

                 INFO     | >> Skipping pretrained video DiT load                                     ]8;id=656631;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py\loader.py]8;;\:]8;id=142373;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py#171\171]8;;\
                          (`skip_dit_load_from_pretrain=True`); initializing video expert randomly                 
                          and expecting checkpoint override.                                                       

05/13 [19:30:57] INFO     | >> Skipping pretrained text encoder/tokenizer load                        ]8;id=537763;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py\loader.py]8;;\:]8;id=878890;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py#206\206]8;;\
                          (`load_text_encoder=False`); training must provide cached                                
                          `context/context_mask`.                                                                  

05/13 [19:31:17] INFO     | >> Finished loading Wan2.2-TI2V-5B components in 118.83 seconds.          ]8;id=416335;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py\loader.py]8;;\:]8;id=897081;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/helpers/loader.py#211\211]8;;\

                 INFO     | >> Skipping ActionDiT pretrained load                                 ]8;id=891815;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/action_dit.py\action_dit.py]8;;\:]8;id=217110;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/action_dit.py#123\123]8;;\
                          (`skip_dit_load_from_pretrain=True`); initializing action expert                         
                          randomly and expecting checkpoint override.                                              

05/13 [19:31:55] INFO     | >> Initialized MoT with experts: ['video', 'action'], num_layers=30           ]8;id=597822;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/mot.py\mot.py]8;;\:]8;id=938351;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/mot.py#53\53]8;;\

                 INFO     | >>   Expert 'video': num_params=5.00 B                                        ]8;id=748559;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/mot.py\mot.py]8;;\:]8;id=978453;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/mot.py#56\56]8;;\

                 INFO     | >>   Expert 'action': num_params=1.02 B                                       ]8;id=85101;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/mot.py\mot.py]8;;\:]8;id=307563;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/models/wan22/mot.py#56\56]8;;\

output_dir: /data_all/xiangchengzhan/FastWAM/evaluate_results/robotwin_openloop_episode/robotwin_uncond_3cam_384_1e-4/20260513_192918/robotwin_uncond_3cam_384


In [13]:
# === Cell 4 / 4：构建 DataLoader + 跑 open-loop 评估 ===
# 终端对应片段：
#   - prepare_openloop_dataloader：HuggingFace / datasets 的 HTTP、"Resolving data files: 100%|..."
#   - run_openloop_evaluation：逐 sample 的推理与指标日志（log_every 等）
# 若只改了 EVAL 数据范围：重跑 Cell 2 再跑本格即可，无需重跑 Cell 3。
# 这里 reload 脚本模块，方便在不重启 kernel 的情况下吃到最新可视化逻辑。

import importlib
import experiments.robotwin.run_robotwin_openloop as openloop_mod

openloop_mod = importlib.reload(openloop_mod)
prepare_openloop_dataloader = openloop_mod.prepare_openloop_dataloader
run_openloop_evaluation = openloop_mod.run_openloop_evaluation

dataloader, processor = prepare_openloop_dataloader(cfg)
run_openloop_evaluation(
    cfg,
    model,
    dataloader,
    processor,
    output_dir=output_dir,
    ckpt_path=ckpt_path,
)

05/13 [19:54:13] INFO     | >> HTTP Request: HEAD                                                   ]8;id=693368;file:///home/xiangchengzhan/anaconda3/envs/fastwam/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=156531;file:///home/xiangchengzhan/anaconda3/envs/fastwam/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\
                          https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parque                
                          t/parquet.py "HTTP/1.1 404 Not Found"                                                    

05/13 [19:59:03] INFO     | >> Using dataset stats:                                      ]8;id=850799;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/datasets/lerobot/robot_video_dataset.py\robot_video_dataset.py]8;;\:]8;id=191915;file:///data_all/xiangchengzhan/FastWAM/src/fastwam/datasets/lerobot/robot_video_dataset.py#104\104]8;;\
                          /data_all/xiangchengzhan/FastWAM/checkpoints/fastwam_release/r                           
                          obotwin_uncond_3cam_384_dataset_stats.json                                               

                 INFO     | >> Open-loop episode subset: split=train episodes=[0]      ]8;id=323411;file:///data_all/xiangchengzhan/FastWAM/experiments/robotwin/run_robotwin_openloop.py\run_robotwin_openloop.py]8;;\:]8;id=10689;file:///data_all/xiangchengzhan/FastWAM/experiments/robotwin/run_robotwin_openloop.py#546\546]8;;\
                          frame_stride=50 samples=4                                                                

05/13 [20:00:36] INFO     | >> Open-loop summary saved to                              ]8;id=478297;file:///data_all/xiangchengzhan/FastWAM/experiments/robotwin/run_robotwin_openloop.py\run_robotwin_openloop.py]8;;\:]8;id=188671;file:///data_all/xiangchengzhan/FastWAM/experiments/robotwin/run_robotwin_openloop.py#727\727]8;;\
                          /data_all/xiangchengzhan/FastWAM/evaluate_results/robotwin_o                             
                          penloop_episode/robotwin_uncond_3cam_384_1e-4/20260513_19291                             
                          8/robotwin_uncond_3cam_384/summary.json                                                  

{
  "ckpt": "/data_all/xiangchengzhan/FastWAM/checkpoints/fastwam_release/robotwin_uncond_3cam_384.pt",
  "split": "train",
  "num_samples": 4,
  "predict_video": true,
  "metrics": {
    "action_l1": 0.00933120702393353,
    "action_mse": 0.00026726952819444705,
    "action_rmse": 0.01561842115096119,
    "action_max_abs": 0.06500540673732758,
    "video_psnr": 33.352972984313965,
    "video_ssim": 0.9504209607839584,
    "video_l1": 0.010360753629356623,
    "video_mse": 0.000850928743602708,
    "action_l1_per_dim": [
      0.004650994669646025,
      0.016719305887818336,
      0.00928504578769207,
      0.01751907914876938,
      0.006000231485813856,
      0.01354349683970213,
      0.005246930755674839,
      0.005027125123888254,
      0.012296434491872787,
      0.00887657143175602,
      0.012840010225772858,
      0.0027542789466679096,
      0.011006336659193039,
      0.004871054086834192
    ],
    "action_mse_per_dim": [
      3.77404285245575e-05,
      0.00085940258577